In [1]:
# Install required packages
!pip install -q transformers datasets torch scikit-learn pandas accelerate scipy seaborn matplotlib
print("📦 Packages installed!")

# Mount and navigate
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/yakut-sa')
print(f"📂 Working directory: {os.getcwd()}")

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import Dataset
import json
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# Reproducibility
# ----------------------------
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ----------------------------
# Config
# ----------------------------
CONFIG = {
    'model_name': "DeepPavlov/rubert-base-cased",
    'use_extended_tokenizer': False,  # Set to True to use extended tokenizer
    'custom_tokenizer_path': "models/extended_rubert_tokenizer",
    'folds_dir': "data/splits/pooled_5fold",
    'outdir': "results_rubert_yakut_baseline",

    # Training parameters
    'epochs': 5,
    'batch_size': 8,
    'lr': 1e-5,
    'max_length': 210,
    'seed': 42,

    # Class balancing
    'use_class_weights': True,

    # Early stopping
    'early_stopping_patience': 2,
    'early_stopping_threshold': 0.001,

    # Training parameters
    'label_smoothing': 0.05,
    'warmup_ratio': 0.06,
    'weight_decay': 0.01,

    # Memory/throughput safety
    'gradient_accumulation_steps': 2,
    'fp16': True,

    # Logging/eval/save
    'logging_steps': 20,
    'eval_strategy': "steps",
    'eval_steps': 40,
    'save_strategy': "steps",
    'save_steps': 40,

    'metric_for_best_model': "eval_f1",
    'greater_is_better': True,
    'load_best_model_at_end': True,
    'save_total_limit': 2,
}
set_seed(CONFIG['seed'])

# ----------------------------
# Fold verification
# ----------------------------
def _label_col(df):
    cols = [c.lower().strip() for c in df.columns]
    if "label" in cols:
        return "label"
    if "sentiment" in cols:
        return "sentiment"
    raise ValueError("No 'label' or 'sentiment' column in dataframe.")

def verify_and_analyze_folds():
    folds_dir = Path(CONFIG['folds_dir'])
    print(f"📊 Analyzing fold structure...")

    fold_stats = {}
    for i in range(1, 6):
        train_file = folds_dir / f"train_fold{i}.csv"
        val_file   = folds_dir / f"val_fold{i}.csv"
        if not train_file.exists() or not val_file.exists():
            print(f"❌ Missing files for fold {i}")
            return False

        train_df = pd.read_csv(train_file)
        val_df   = pd.read_csv(val_file)
        t_labcol = _label_col(train_df)
        v_labcol = _label_col(val_df)

        fold_stats[i] = {
            'train_size': len(train_df),
            'val_size': len(val_df),
            'train_first_text': train_df.iloc[0]['text'][:50] if len(train_df) > 0 else "",
            'val_first_text': val_df.iloc[0]['text'][:50] if len(val_df) > 0 else "",
            'train_labels': train_df[t_labcol].value_counts().to_dict(),
            'val_labels': val_df[v_labcol].value_counts().to_dict()
        }

    unique_checks = {fold_stats[i]['train_first_text'] for i in range(1, 6)}
    if len(unique_checks) == 1:
        print("⚠️ WARNING: All folds appear to have the same data!")
    else:
        print(f"✅ Found {len(unique_checks)} unique fold configurations")

    for fold_num, stats in fold_stats.items():
        print(f"\nFold {fold_num}:")
        print(f"  Train: {stats['train_size']} samples - {stats['train_labels']}")
        print(f"  Val:   {stats['val_size']} samples - {stats['val_labels']}")

    return True

# ----------------------------
# Data loading / prep
# ----------------------------
def load_and_prepare_fold(fold_num: int, tokenizer, label_encoder=None):
    folds_dir = Path(CONFIG['folds_dir'])

    train_df = pd.read_csv(folds_dir / f"train_fold{fold_num}.csv")
    val_df   = pd.read_csv(folds_dir / f"val_fold{fold_num}.csv")

    t_labcol = _label_col(train_df)
    v_labcol = _label_col(val_df)

    if label_encoder is None:
        unique_labels = sorted(set(train_df[t_labcol].unique()) | set(val_df[v_labcol].unique()))
        label_encoder = {label: idx for idx, label in enumerate(unique_labels)}

    train_df = train_df[['text', t_labcol]].dropna().copy()
    val_df   = val_df[['text', v_labcol]].dropna().copy()
    train_df['label'] = train_df[t_labcol].map(label_encoder)
    val_df['label']   = val_df[v_labcol].map(label_encoder)

    if train_df['label'].isna().any() or val_df['label'].isna().any():
        bad = set(train_df[t_labcol].unique()) | set(val_df[v_labcol].unique())
        raise ValueError(f"Unknown labels in fold {fold_num}: {bad}")

    class_weights = None
    if CONFIG['use_class_weights']:
        labels = train_df['label'].astype(int).values
        classes = np.unique(labels)
        weights = compute_class_weight('balanced', classes=classes, y=labels)
        class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
        print(f"📊 Class weights for fold {fold_num}: {class_weights}")

    return train_df[['text','label']], val_df[['text','label']], label_encoder, class_weights

# ----------------------------
# Tokenization
# ----------------------------
def tokenize_dataset(df: pd.DataFrame, tokenizer, max_length: int):
    def tokenize_function(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            return_tensors=None
        )

    df_copy = df.copy()
    df_copy['label'] = df_copy['label'].astype(int)
    dataset = Dataset.from_pandas(df_copy[['text', 'label']], preserve_index=False)
    tokenized = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=['text'],
        desc="Tokenizing"
    )
    return tokenized

# ----------------------------
# Trainer with class weights
# ----------------------------
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs[0]

        if self.class_weights is not None:
            num_labels = model.config.num_labels
            weight_vec = torch.ones(num_labels, dtype=torch.float32, device=logits.device)
            for cls_id, w in self.class_weights.items():
                if 0 <= int(cls_id) < num_labels:
                    weight_vec[int(cls_id)] = float(w)

            loss_fct = torch.nn.CrossEntropyLoss(
                weight=weight_vec,
                label_smoothing=self.args.label_smoothing_factor if hasattr(self.args, "label_smoothing_factor") else 0.0,
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss(
                label_smoothing=self.args.label_smoothing_factor if hasattr(self.args, "label_smoothing_factor") else 0.0,
            )

        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ----------------------------
# One-fold training
# ----------------------------
def train_fold(fold_num: int, tokenizer, global_label_encoder=None):
    print(f"\n{'='*60}\nTRAINING FOLD {fold_num}\n{'='*60}")

    train_df, val_df, label_encoder, class_weights = load_and_prepare_fold(
        fold_num, tokenizer, global_label_encoder
    )
    if global_label_encoder is None:
        global_label_encoder = label_encoder

    num_labels = len(label_encoder)
    reverse_encoder = {v: k for k, v in label_encoder.items()}

    print(f"\n📊 Label distribution (fold {fold_num}):")
    for label_id in sorted(reverse_encoder.keys()):
        train_count = int((train_df['label'] == label_id).sum())
        val_count   = int((val_df['label']   == label_id).sum())
        print(f"  {reverse_encoder[label_id]}: Train={train_count}, Val={val_count}")

    print(f"\n🤖 Loading model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model_name'],
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )

    # Only resize if using extended tokenizer
    if CONFIG['use_extended_tokenizer'] and len(tokenizer) != model.config.vocab_size:
        print(f"🔧 Resizing embeddings: {model.config.vocab_size} → {len(tokenizer)}")
        model.resize_token_embeddings(len(tokenizer))

    print(f"🔄 Tokenizing datasets...")
    train_dataset = tokenize_dataset(train_df, tokenizer, CONFIG['max_length'])
    eval_dataset  = tokenize_dataset(val_df, tokenizer, CONFIG['max_length'])

    output_dir = Path(CONFIG['outdir']) / f"fold_{fold_num}"
    output_dir.mkdir(exist_ok=True, parents=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=CONFIG['epochs'],
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=max(CONFIG['batch_size'], 16),
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],

        learning_rate=CONFIG['lr'],
        warmup_ratio=CONFIG['warmup_ratio'],
        weight_decay=CONFIG['weight_decay'],

        eval_strategy=CONFIG['eval_strategy'],
        eval_steps=CONFIG['eval_steps'],
        save_strategy=CONFIG['save_strategy'],
        save_steps=CONFIG['save_steps'],

        logging_steps=CONFIG['logging_steps'],
        save_total_limit=CONFIG['save_total_limit'],
        load_best_model_at_end=CONFIG['load_best_model_at_end'],
        metric_for_best_model=CONFIG['metric_for_best_model'],
        greater_is_better=CONFIG['greater_is_better'],

        fp16=CONFIG['fp16'] and torch.cuda.is_available(),
        label_smoothing_factor=CONFIG['label_smoothing'],

        report_to=[],
        seed=CONFIG['seed'],
        push_to_hub=False,

        gradient_checkpointing=True,
        tf32=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    )

    def compute_metrics(eval_pred):
        if isinstance(eval_pred, tuple):
            preds, labels = eval_pred
        else:
            preds, labels = eval_pred.predictions, eval_pred.label_ids
        if isinstance(preds, (list, tuple)):
            preds = preds[0]
        predictions = np.argmax(preds, axis=1)

        accuracy = accuracy_score(labels, predictions)
        f1_macro = f1_score(labels, predictions, average='macro')
        f1_weighted = f1_score(labels, predictions, average='weighted')
        return {"accuracy": accuracy, "f1": f1_weighted, "f1_macro": f1_macro}

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        class_weights=class_weights,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=CONFIG['early_stopping_patience'],
            early_stopping_threshold=CONFIG['early_stopping_threshold']
        )]
    )

    print(f"\n🚀 Starting training...")
    train_result = trainer.train()

    print(f"\n📊 Evaluating...")
    eval_result = trainer.evaluate()

    predictions = trainer.predict(eval_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    target_names = [reverse_encoder[i] for i in sorted(reverse_encoder.keys())]
    print(f"\n📈 Classification Report (Fold {fold_num}):")
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

    if CONFIG['load_best_model_at_end']:
        trainer.save_model(str(output_dir / "best_model"))
        tokenizer.save_pretrained(str(output_dir / "best_model"))

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'fold': fold_num,
        'accuracy': float(eval_result['eval_accuracy']),
        'f1': float(eval_result['eval_f1']),
        'f1_macro': float(eval_result['eval_f1_macro']),
        'predictions': y_pred.tolist(),
        'labels': y_true.tolist()
    }

# ----------------------------
# Main
# ----------------------------
def main():
    tokenizer_type = "EXTENDED" if CONFIG['use_extended_tokenizer'] else "BASELINE"
    print(f"🚀 Starting Yakut RuBERT Training - {tokenizer_type} TOKENIZER")
    print("="*60)

    if not verify_and_analyze_folds():
        raise ValueError("Data verification failed!")

    print(f"\n🔧 Loading tokenizer...")
    if CONFIG['use_extended_tokenizer']:
        tokenizer_path = CONFIG['custom_tokenizer_path']
        if Path(tokenizer_path).exists():
            tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True)
            print(f"✅ Extended tokenizer loaded: {len(tokenizer):,} tokens")
        else:
            raise FileNotFoundError(f"Extended tokenizer not found at {tokenizer_path}")
    else:
        tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], use_fast=True)
        print(f"✅ Baseline tokenizer loaded: {len(tokenizer):,} tokens")

    train_df = pd.read_csv(Path(CONFIG['folds_dir']) / "train_fold1.csv")
    val_df   = pd.read_csv(Path(CONFIG['folds_dir']) / "val_fold1.csv")
    t_labcol = _label_col(train_df)
    v_labcol = _label_col(val_df)
    all_labels = sorted(set(train_df[t_labcol].unique()) | set(val_df[v_labcol].unique()))
    global_label_encoder = {label: idx for idx, label in enumerate(all_labels)}
    print(f"\n📊 Label encoding: {global_label_encoder}")

    all_results = []
    for fold in range(1, 6):
        try:
            result = train_fold(fold, tokenizer, global_label_encoder)
            all_results.append(result)
            print(f"✅ Fold {fold} completed: Acc={result['accuracy']:.4f}, F1={result['f1']:.4f}")
        except Exception as e:
            print(f"❌ Fold {fold} failed: {e}")
            import traceback; traceback.print_exc()

    if all_results:
        accuracies = [r['accuracy'] for r in all_results]
        f1_scores = [r['f1'] for r in all_results]
        f1_macro_scores = [r['f1_macro'] for r in all_results]

        print(f"\n{'='*60}\nFINAL CROSS-VALIDATION RESULTS\n{'='*60}")
        print(f"📊 Accuracy:    {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
        print(f"📊 F1-Weighted: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
        print(f"📊 F1-Macro:    {np.mean(f1_macro_scores):.4f} ± {np.std(f1_macro_scores):.4f}")

        results_file = Path(CONFIG['outdir']) / 'cv_results.json'
        results_file.parent.mkdir(exist_ok=True, parents=True)
        with open(results_file, 'w') as f:
            json.dump({
                'config': CONFIG,
                'results': all_results,
                'summary': {
                    'accuracy_mean': float(np.mean(accuracies)),
                    'accuracy_std': float(np.std(accuracies)),
                    'f1_weighted_mean': float(np.mean(f1_scores)),
                    'f1_weighted_std': float(np.std(f1_scores)),
                    'f1_macro_mean': float(np.mean(f1_macro_scores)),
                    'f1_macro_std': float(np.std(f1_macro_scores))
                }
            }, f, indent=2)
        print(f"\n💾 Results saved to {results_file}")

if __name__ == "__main__":
    main()

📦 Packages installed!
Mounted at /content/drive
📂 Working directory: /content/drive/MyDrive/yakut-sa
🚀 Starting Yakut RuBERT Training - BASELINE TOKENIZER
📊 Analyzing fold structure...
✅ Found 2 unique fold configurations

Fold 1:
  Train: 638 samples - {'neutral': 347, 'positive': 152, 'negative': 139}
  Val:   160 samples - {'neutral': 87, 'positive': 38, 'negative': 35}

Fold 2:
  Train: 638 samples - {'neutral': 347, 'positive': 152, 'negative': 139}
  Val:   160 samples - {'neutral': 87, 'positive': 38, 'negative': 35}

Fold 3:
  Train: 638 samples - {'neutral': 347, 'positive': 152, 'negative': 139}
  Val:   160 samples - {'neutral': 87, 'positive': 38, 'negative': 35}

Fold 4:
  Train: 639 samples - {'neutral': 348, 'positive': 152, 'negative': 139}
  Val:   159 samples - {'neutral': 86, 'positive': 38, 'negative': 35}

Fold 5:
  Train: 639 samples - {'neutral': 347, 'positive': 152, 'negative': 140}
  Val:   159 samples - {'neutral': 87, 'positive': 38, 'negative': 34}

🔧 Loadi

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Tokenizing:   0%|          | 0/159 [00:00<?, ? examples/s]


🚀 Starting training...


Step,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
40,1.121500,1.080776,0.327044,0.284188,0.320302
80,1.067400,1.045969,0.421384,0.429389,0.414235
120,1.014900,1.027260,0.484277,0.488735,0.475869
160,0.973100,1.021808,0.503145,0.505166,0.499939
200,0.940100,1.028188,0.452830,0.463724,0.439171



📊 Evaluating...



📈 Classification Report (Fold 5):
              precision    recall  f1-score   support

    negative     0.4528    0.7059    0.5517        34
     neutral     0.6727    0.4253    0.5211        87
    positive     0.3725    0.5000    0.4270        38

    accuracy                         0.5031       159
   macro avg     0.4994    0.5437    0.4999       159
weighted avg     0.5540    0.5031    0.5052       159

✅ Fold 5 completed: Acc=0.5031, F1=0.5052

FINAL CROSS-VALIDATION RESULTS
📊 Accuracy:    0.4938 ± 0.0268
📊 F1-Weighted: 0.4919 ± 0.0295
📊 F1-Macro:    0.4502 ± 0.0436

💾 Results saved to results_rubert_yakut_baseline/cv_results.json
